# Proyecto Final Big Data — Siniestralidad vial en Bogotá
## Cuaderno 1 · Recolección, perfilado, limpieza y modelado (Apache Spark)

**Fuente:** Datos Abiertos Bogotá (Secretaría Distrital de Movilidad), dataset `historico-siniestros-bogota-d-c`
(https://datosabiertos.bogota.gov.co/dataset/historico-siniestros-bogota-d-c). Licencia CC BY 4.0.
El CSV completo se descargó manualmente desde el portal y se sube a la sesión de Colab (`/content/historico_siniestros_bogota.csv`).

**Fases que cubre este cuaderno:** (1) recolección y verificación, (2) perfilado, (3) limpieza y variables derivadas con Spark, (4) modelo dimensional, (5) capa limpia en Parquet y reporte de calidad.

**Salida:** `interim_parquet.zip` (se descarga al final) — es la entrada de los cuadernos 2 y 3.

Hallazgos del perfilado que motivan la limpieza:
- `LATITUD`/`LONGITUD` (y sus copias `X`/`Y`) vienen como texto con puntos de miles mal ubicados (`-74.090.923.953`, `469.380.685`), un artefacto de exportación. Se reconstruyen quitando los puntos y reinsertando el decimal tras 1 dígito (latitud) o 2 (longitud).
- `FECHA_HORA_ACC` trae un sufijo `+00` que parece artefacto: la distribución horaria (picos en horas laborales) indica que son horas locales de Bogotá, no UTC. Se tratan como hora local (inferencia, no confirmada por el portal).
- `PK_CALZADA` tiene muchos nulos; se conserva como atributo opcional.

## 0. Rutas y detección de formato

In [ ]:
from pathlib import Path

CSV_LOCAL = '/content/historico_siniestros_bogota.csv'   # sube el CSV al panel de archivos de Colab
BASE = Path('/content/proyecto_bigdata')
INTERIM = BASE / 'interim'
OUT = BASE / 'out'
for d in (INTERIM, OUT):
    d.mkdir(parents=True, exist_ok=True)

if not Path(CSV_LOCAL).exists():
    from google.colab import files
    print('No encuentro el CSV: súbelo ahora (historico_siniestros_bogota.csv)')
    subido = files.upload()
    Path(list(subido)[0]).rename(CSV_LOCAL)

def detectar_formato(path):
    crudo = open(path, 'rb').read(200_000)
    try:
        crudo.decode('utf-8'); enc = 'UTF-8'
    except UnicodeDecodeError:
        enc = 'ISO-8859-1'
    primera = crudo.decode(enc, errors='ignore').lstrip('\ufeff').split('\n', 1)[0]
    sep = max([',', ';', '|', '\t'], key=primera.count)
    return enc, sep

enc, sep = detectar_formato(CSV_LOCAL)
print(Path(CSV_LOCAL).name, f'{Path(CSV_LOCAL).stat().st_size/1e6:.1f} MB | encoding: {enc} | separador: {sep!r}')

## 1. Verificación del archivo recolectado

In [ ]:
import hashlib, datetime as dt, pandas as pd

def sha256(path, bloque=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(bloque):
            h.update(chunk)
    return h.hexdigest()

pd.Series({
    'archivo': Path(CSV_LOCAL).name,
    'size_mb': round(Path(CSV_LOCAL).stat().st_size / 1e6, 1),
    'sha256': sha256(CSV_LOCAL),
    'fuente': 'datosabiertos.bogota.gov.co/dataset/historico-siniestros-bogota-d-c',
    'licencia': 'CC BY 4.0',
    'verificado': dt.date.today().isoformat(),
})

## 2. Perfilado exploratorio (pandas): columnas, tipos, nulos y muestra

In [ ]:
pdf = pd.read_csv(CSV_LOCAL, encoding='utf-8-sig' if enc == 'UTF-8' else 'latin-1', sep=sep,
                  low_memory=False, dtype=str)
print('filas, columnas:', pdf.shape)

perfil = pd.DataFrame({
    'nulos_%': (pdf.isna().mean() * 100).round(1),
    'distintos': pdf.nunique(dropna=True),
    'ejemplo': pdf.iloc[0].astype(str).str.slice(0, 40),
})
pd.set_option('display.max_rows', 200, 'display.width', 200)
print(perfil.to_string())
pdf.head(5)

In [ ]:
for c in ('GRAVEDAD', 'CLASE_ACC', 'ANO_OCURRENCIA_ACC'):
    print(c, pdf[c].value_counts(dropna=False).to_dict())
print('Localidades distintas:', pdf['LOCALIDAD'].nunique())
del pdf   # liberar memoria antes de pasar a Spark

---
# Limpieza y modelado con Apache Spark

## 0. Spark

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pathlib import Path

spark = (SparkSession.builder
         .master('local[*]')
         .appName('siniestros_bogota')
         .config('spark.sql.session.timeZone', 'America/Bogota')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)

# Cuando integremos HDFS, solo cambia esta URI (hdfs://localhost:9000/...)
CSV_LOCAL = '/content/historico_siniestros_bogota.csv'
RAW_URI = f'file://{CSV_LOCAL}'
BASE = Path('/content/proyecto_bigdata')
INTERIM = BASE / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)

## 1. Lectura de la capa cruda

In [ ]:
raw = (spark.read
       .option('header', True)
       .option('encoding', enc)
       .option('sep', sep)
       .csv(RAW_URI))
# limpia posibles BOM/espacios en los nombres de columnas
raw = raw.toDF(*[c.replace('\ufeff', '').strip() for c in raw.columns])

n_raw = raw.count()
print('filas crudas:', n_raw, '| columnas:', len(raw.columns))
print(raw.columns)
raw.printSchema()

## 2. Reconstrucción de coordenadas
Se valida primero que `X`/`Y` sean copias de `LONGITUD`/`LATITUD`; si lo son, se descartan.

In [ ]:
def num_desde_texto(col, enteros):
    s = F.regexp_replace(F.col(col), r'\.', '')
    signo = F.when(s.startswith('-'), F.lit(-1.0)).otherwise(F.lit(1.0))
    d = F.regexp_replace(s, '-', '')
    return signo * F.concat(F.substring(d, 1, enteros), F.lit('.'),
                            F.substring(d, enteros + 1, 30)).cast('double')

df = (raw
      .withColumn('latitud', num_desde_texto('LATITUD', 1))
      .withColumn('longitud', num_desde_texto('LONGITUD', 2)))

chk = (df.withColumn('x_', num_desde_texto('X', 2))
         .withColumn('y_', num_desde_texto('Y', 1)))
x_igual = chk.filter(F.abs(F.col('x_') - F.col('longitud')) > 1e-6).count() == 0
y_igual = chk.filter(F.abs(F.col('y_') - F.col('latitud')) > 1e-6).count() == 0
print('X == LONGITUD:', x_igual, '| Y == LATITUD:', y_igual)

## 3. Limpieza y variables derivadas
Coordenadas fuera de Bogotá (lat 4.40–4.90, lon −74.30 a −73.90) se marcan como inválidas y se anulan, sin descartar el registro.

In [ ]:
bbox = F.col('latitud').between(4.40, 4.90) & F.col('longitud').between(-74.30, -73.90)

df = (df
  .withColumn('coord_valida', bbox)
  .withColumn('latitud', F.when(F.col('coord_valida'), F.col('latitud')))
  .withColumn('longitud', F.when(F.col('coord_valida'), F.col('longitud')))
  .withColumn('fecha_hora', F.to_timestamp(F.regexp_replace('FECHA_HORA_ACC', r'\+00$', ''), 'yyyy/MM/dd HH:mm:ss'))
  .withColumn('fecha', F.to_date('fecha_hora'))
  .withColumn('anio', F.year('fecha_hora'))
  .withColumn('mes', F.month('fecha_hora'))
  .withColumn('dow', F.dayofweek('fecha_hora'))
  .withColumn('hora', F.hour('fecha_hora'))
  .withColumn('franja', F.when(F.col('hora').between(0, 5), 'MADRUGADA')
                         .when(F.col('hora').between(6, 11), 'MANANA')
                         .when(F.col('hora').between(12, 17), 'TARDE')
                         .otherwise('NOCHE'))
  .withColumn('fin_de_semana', F.col('dow').isin(1, 7))
  .withColumn('gravedad', F.upper(F.trim('GRAVEDAD')))
  .withColumn('clase_acc', F.upper(F.trim('CLASE_ACC')))
  .withColumn('localidad', F.upper(F.trim('LOCALIDAD')))
  .withColumn('es_grave', F.col('gravedad').isin('CON HERIDOS', 'CON MUERTOS'))
  .withColumn('codigo_accidente', F.col('CODIGO_ACCIDENTE').cast('long'))
  .withColumn('civ', F.col('CIV').cast('double').cast('long'))
  .withColumn('pk_calzada', F.col('PK_CALZADA').cast('double').cast('long'))
  .withColumn('anio_consistente', F.col('anio') == F.col('ANO_OCURRENCIA_ACC').cast('int')))

n_anio_incons = df.filter(~F.col('anio_consistente')).count()
n_fecha_nula = df.filter(F.col('fecha_hora').isNull()).count()
n_coord_inval = df.filter(~F.col('coord_valida')).count()
print('año inconsistente:', n_anio_incons, '| fecha nula:', n_fecha_nula, '| coordenadas inválidas:', n_coord_inval)

## 4. Deduplicación por código de accidente

In [ ]:
w = Window.partitionBy('codigo_accidente').orderBy('fecha_hora')
dedup = df.withColumn('rn', F.row_number().over(w)).filter('rn = 1').drop('rn')
n_dedup = dedup.count()
print(f'filas: {n_raw} -> {n_dedup} (duplicados eliminados: {n_raw - n_dedup})')

## 5. Modelo dimensional (tabla de hechos + dimensiones)

In [ ]:
fact = dedup.select(
    'codigo_accidente', F.col('FORMULARIO').alias('formulario'),
    'fecha_hora', 'fecha', 'anio', 'mes', 'hora', 'franja', 'fin_de_semana',
    'localidad', 'clase_acc', 'gravedad', 'es_grave',
    F.col('DIRECCION').alias('direccion'),
    'latitud', 'longitud', 'coord_valida', 'civ', 'pk_calzada')

dim_localidad = fact.select('localidad').distinct().withColumn('localidad_id', F.dense_rank().over(Window.orderBy('localidad')))
dim_clase = fact.select('clase_acc').distinct().withColumn('clase_id', F.dense_rank().over(Window.orderBy('clase_acc')))
dim_gravedad = (fact.select('gravedad').distinct()
                .withColumn('gravedad_id', F.dense_rank().over(Window.orderBy('gravedad')))
                .withColumn('nivel', F.when(F.col('gravedad') == 'SOLO DANOS', 0)
                                      .when(F.col('gravedad') == 'CON HERIDOS', 1).otherwise(2)))
dim_tiempo = (fact.select('fecha', 'anio', 'mes', 'fin_de_semana').distinct()
              .withColumn('dia_semana', F.date_format('fecha', 'E')))

hechos = (fact
          .join(dim_localidad, 'localidad', 'left')
          .join(dim_clase, 'clase_acc', 'left')
          .join(dim_gravedad.select('gravedad', 'gravedad_id'), 'gravedad', 'left')
          .select('codigo_accidente', 'formulario', 'fecha_hora', 'fecha', 'anio', 'mes', 'hora', 'franja',
                  'localidad_id', 'clase_id', 'gravedad_id', 'es_grave',
                  'direccion', 'latitud', 'longitud', 'coord_valida', 'civ', 'pk_calzada'))

for nombre, tabla in {'dim_localidad': dim_localidad, 'dim_clase': dim_clase,
                      'dim_gravedad': dim_gravedad, 'dim_tiempo': dim_tiempo}.items():
    print(f'{nombre}: {tabla.count()} filas')
hechos.show(3, truncate=False)

## 6. Escritura en Parquet (capa limpia)

In [ ]:
salida = f'file://{INTERIM}'
fact.write.mode('overwrite').partitionBy('anio').parquet(f'{salida}/siniestros_limpio')
hechos.write.mode('overwrite').partitionBy('anio').parquet(f'{salida}/fact_siniestros')
for nombre, tabla in {'dim_localidad': dim_localidad, 'dim_clase': dim_clase,
                      'dim_gravedad': dim_gravedad, 'dim_tiempo': dim_tiempo}.items():
    tabla.write.mode('overwrite').parquet(f'{salida}/{nombre}')

print('Verificación de lectura:', spark.read.parquet(f'{salida}/fact_siniestros').count(), 'filas')

## 7. Reporte de calidad de datos

In [ ]:
import json, pandas as pd

reporte = {
    'filas_crudas': n_raw,
    'filas_limpias': n_dedup,
    'duplicados_eliminados': n_raw - n_dedup,
    'fechas_nulas': n_fecha_nula,
    'coordenadas_invalidas': n_coord_inval,
    'anio_inconsistente': n_anio_incons,
    'xy_duplican_lonlat': bool(x_igual and y_igual),
}
(BASE / 'out').mkdir(exist_ok=True)
(BASE / 'out' / 'reporte_calidad.json').write_text(json.dumps(reporte, indent=2))
display(pd.Series(reporte, name='valor').to_frame())

print('\nDistribución por gravedad, clase, año y franja:')
for c in ('gravedad', 'clase_acc', 'anio', 'franja'):
    display(fact.groupBy(c).count().orderBy(c).toPandas())

## 8. Empaquetar la capa limpia para los cuadernos 2 y 3

In [ ]:
import zipfile

zpath = BASE / 'interim_parquet.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(INTERIM.rglob('*')):
        if f.is_file() and not f.name.endswith('.crc'):
            z.write(f, f.relative_to(BASE))
    z.write(BASE / 'out' / 'reporte_calidad.json', 'out/reporte_calidad.json')
print(zpath, f'{zpath.stat().st_size/1e6:.2f} MB')

from google.colab import files
files.download(str(zpath))